In [ ]:
import torch
from datasets import Dataset
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

# Load your dataset from CSV files without a header row
train_data_path = 'data/train_2020_2022_2024.csv'
test_data_path = 'data/test_2020_2022_2024.csv'
test_preds_path = 'data/test_predictions_bert_2024.csv'
model_dir = 'models/'

# Define the column names since there's no header row
columns = ['ad_id', 'text', 'DONATE', 'CONTACT', 'PURCHASE', 'GOTV', 'EVENT', 'POLL', 'GATHERINFO', 'LEARNMORE', "PRIMARY_PERSUADE"]
goals = ["DONATE", "CONTACT", "PURCHASE", "GOTV", "EVENT", "POLL", "GATHERINFO", "LEARNMORE", "PRIMARY_PERSUADE"]

# Read the CSV files
train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)
train_df = train_df.rename(columns={"combined_everything": "text"})
test_df = test_df.rename(columns={"combined_everything": "text"})

test_predictions = {}

for goal in goals:
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    train_dataset = train_dataset.rename_column(goal, 'label')
    test_dataset = test_dataset.rename_column(goal, 'label')

    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

    def tokenize_function(examples):
        return tokenizer(examples['text'], truncation=True, padding=False)

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)

    # Remove columns the Trainer doesn't need
    keep_cols = {"input_ids", "attention_mask", "label"}
    train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep_cols])
    test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in keep_cols])

    train_dataset.set_format("torch")
    test_dataset.set_format("torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir="./temp",
        per_device_train_batch_size=32,
        num_train_epochs=3,
        eval_strategy="epoch",        # replaces evaluate_during_training
        logging_steps=100,
        logging_dir="./logs",
        report_to="none",
    )

    def compute_metrics(p):
        predictions, labels = p.predictions, p.label_ids
        predictions = np.argmax(predictions, axis=1)
        f1 = f1_score(labels, predictions, average="weighted")
        return {"f1_score": f1}

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    results = trainer.evaluate(test_dataset)
    print(f"{goal} — F1 Score: {results['eval_f1_score']:.4f}")

    # Predict on test set
    data_loader = trainer.get_test_dataloader(test_dataset)
    all_predictions = []
    model.eval()
    for batch in data_loader:
        inputs = batch["input_ids"].to(trainer.args.device)
        attention_mask = batch["attention_mask"].to(trainer.args.device)
        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
        all_predictions.extend(preds)

    test_predictions[goal] = np.array(all_predictions)
    trainer.save_model(model_dir + goal)

# Save test set predictions
df_test_preds = pd.DataFrame(test_predictions)
df_test_preds['ad_id'] = test_df['ad_id']
df_test_preds.to_csv(test_preds_path, index=False)